In [1]:
import math
import re
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

# Lab2 notebook is inside LLM/Lab2.
# Therefore ../Lab1/data points to the dataset created in Lab1.
DATA_DIR = Path("../Lab1/data")

TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "val.jsonl"
TEST_PATH = DATA_DIR / "test.jsonl"

for path in (TRAIN_PATH, VAL_PATH, TEST_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path.resolve()}")

print("Using dataset:", DATA_DIR.resolve())
print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)


Using dataset: /Users/loki/Desktop/mine/Labs/LLM/Lab1/data
Train: ../Lab1/data/train.jsonl
Validation: ../Lab1/data/val.jsonl
Test: ../Lab1/data/test.jsonl


In [2]:
# ============================================================
# Feature extraction — same feature set as Assignment 1
# ============================================================

_WORD_RE = re.compile(r"\S+")

def _words(text):
    return _WORD_RE.findall(text)

def _word_ngrams(words, n):
    if len(words) < n:
        return []
    return ["_".join(words[i:i + n]) for i in range(len(words) - n + 1)]

def _char_ngrams(text, n):
    text = text.replace(" ", "")
    if len(text) < n:
        return []
    return [text[i:i + n] for i in range(len(text) - n + 1)]

def extract_features(text):
    text = text.strip().lower()
    words = _words(text)

    feats = []

    # Word unigrams and bigrams
    feats += ["W1|" + w for w in _word_ngrams(words, 1)]
    feats += ["W2|" + w for w in _word_ngrams(words, 2)]

    # Character 2/3/4-grams
    for n in (2, 3, 4):
        feats += [f"C{n}|" + g for g in _char_ngrams(text, n)]

    return feats


In [3]:
# ============================================================
# Load JSONL dataset
# ============================================================

def load_jsonl(path):
    texts, labels = [], []

    with open(path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            texts.append(row["text"])
            labels.append(row["label"])

    return texts, labels


train_texts, train_labels = load_jsonl(TRAIN_PATH)
val_texts, val_labels = load_jsonl(VAL_PATH)
test_texts, test_labels = load_jsonl(TEST_PATH)

print("Train samples:", len(train_texts))
print("Validation samples:", len(val_texts))
print("Test samples:", len(test_texts))

print("\nLanguages:", sorted(set(train_labels)))
print("Number of languages:", len(set(train_labels)))


Train samples: 18400
Validation samples: 2300
Test samples: 2300

Languages: ['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'khasi', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'santhali', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']
Number of languages: 23


In [4]:
# ============================================================
# Logistic Regression — same model as Assignment 1
# ============================================================

class LogisticRegression:
    def __init__(
        self,
        n_features,
        n_classes,
        lr=0.5,
        epochs=20,
        batch_size=64,
        l2=1e-4,
        verbose=True,
        seed=0
    ):
        self.n_features = n_features
        self.n_classes = n_classes
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2
        self.verbose = verbose

        random.seed(seed)

        self.W = [[0.0] * n_classes for _ in range(n_features)]
        self.b = [0.0] * n_classes

    def _logits(self, sparse_vec):
        logits = self.b[:]

        for idx, val in sparse_vec.items():
            row = self.W[idx]

            for k in range(self.n_classes):
                logits[k] += val * row[k]

        return logits

    @staticmethod
    def _softmax(logits):
        m = max(logits)
        exps = [math.exp(z - m) for z in logits]
        s = sum(exps)

        return [e / s for e in exps]

    def _one_hot(self, label):
        y = [0.0] * self.n_classes
        y[label] = 1.0
        return y

    def fit(self, X, y, X_val=None, y_val=None):
        n = len(X)
        indices = list(range(n))

        for epoch in range(self.epochs):
            random.shuffle(indices)
            total_loss = 0.0

            for start in range(0, n, self.batch_size):
                batch = indices[start:start + self.batch_size]
                bsz = len(batch)

                grad_W_updates = {}
                grad_b = [0.0] * self.n_classes

                for i in batch:
                    sparse_vec = X[i]
                    label = y[i]

                    logits = self._logits(sparse_vec)
                    probs = self._softmax(logits)
                    target = self._one_hot(label)

                    eps = 1e-12
                    total_loss += -math.log(probs[label] + eps)

                    diff = [
                        probs[k] - target[k]
                        for k in range(self.n_classes)
                    ]

                    for k in range(self.n_classes):
                        grad_b[k] += diff[k]

                    for idx, val in sparse_vec.items():
                        if idx not in grad_W_updates:
                            grad_W_updates[idx] = [0.0] * self.n_classes

                        row_grad = grad_W_updates[idx]

                        for k in range(self.n_classes):
                            row_grad[k] += val * diff[k]

                for idx, row_grad in grad_W_updates.items():
                    row = self.W[idx]

                    for k in range(self.n_classes):
                        grad = row_grad[k] / bsz + self.l2 * row[k]
                        row[k] -= self.lr * grad

                for k in range(self.n_classes):
                    self.b[k] -= self.lr * (grad_b[k] / bsz)

            if self.verbose:
                msg = (
                    f"epoch {epoch + 1:>3}/{self.epochs}  "
                    f"train_loss={total_loss / n:.4f}"
                )

                if X_val is not None:
                    preds = self.predict(X_val)
                    acc = (
                        sum(
                            1 for p, t in zip(preds, y_val)
                            if p == t
                        )
                        / len(y_val)
                    )

                    msg += f"  val_acc={acc:.4f}"

                print(msg)

        return self

    def predict_proba(self, X):
        return [
            self._softmax(self._logits(x))
            for x in X
        ]

    def predict(self, X):
        preds = []

        for x in X:
            logits = self._logits(x)

            best_k = max(
                range(self.n_classes),
                key=lambda k: logits[k]
            )

            preds.append(best_k)

        return preds


In [5]:
# ============================================================
# Evaluation metrics
# ============================================================

def confusion_counts(y_true, y_pred, n_classes):
    tp = [0] * n_classes
    fp = [0] * n_classes
    fn = [0] * n_classes

    for t, p in zip(y_true, y_pred):
        if t == p:
            tp[t] += 1
        else:
            fp[p] += 1
            fn[t] += 1

    return tp, fp, fn


def precision_recall_f1_per_class(y_true, y_pred, n_classes):
    tp, fp, fn = confusion_counts(
        y_true,
        y_pred,
        n_classes
    )

    precision = [0.0] * n_classes
    recall = [0.0] * n_classes
    f1 = [0.0] * n_classes

    for k in range(n_classes):
        denom_p = tp[k] + fp[k]
        denom_r = tp[k] + fn[k]

        precision[k] = (
            tp[k] / denom_p
            if denom_p > 0
            else 0.0
        )

        recall[k] = (
            tp[k] / denom_r
            if denom_r > 0
            else 0.0
        )

        denom_f = precision[k] + recall[k]

        f1[k] = (
            2 * precision[k] * recall[k] / denom_f
            if denom_f > 0
            else 0.0
        )

    return precision, recall, f1


def macro_f1(y_true, y_pred, n_classes):
    _, _, f1 = precision_recall_f1_per_class(
        y_true,
        y_pred,
        n_classes
    )

    return sum(f1) / len(f1)


def accuracy(y_true, y_pred):
    correct = sum(
        1 for t, p in zip(y_true, y_pred)
        if t == p
    )

    return correct / len(y_true)


In [6]:
# ============================================================
# TF functions
# ============================================================

def tf_un(counts, text):
    """Unnormalized TF: raw term frequency."""
    return dict(counts)


def tf_nt1(counts, text):
    """
    TF normalized by total number of words in the sentence.

    TF(t,d) = count(t,d) / total_words
    """
    words = _words(text)
    total_words = len(words) or 1

    return {
        term: count / total_words
        for term, count in counts.items()
    }


def tf_nt2(counts, text):
    """
    TF normalized by the frequency of the most frequent word.

    TF(t,d) = count(t,d) / maximum_word_frequency
    """
    words = _words(text)

    if not words:
        return dict(counts)

    word_counts = Counter(words)
    max_word_frequency = max(word_counts.values())

    return {
        term: count / max_word_frequency
        for term, count in counts.items()
    }


TF_FUNCS = {
    "un": tf_un,
    "nt1": tf_nt1,
    "nt2": tf_nt2
}


In [7]:
# ============================================================
# IDF functions
# ============================================================

def idf_un(df, n_docs):
    """
    Unnormalized smoothed IDF.

    IDF(t) = log((1 + N) / (1 + DF(t))) + 1
    """
    return {
        term: math.log((1 + n_docs) / (1 + d)) + 1.0
        for term, d in df.items()
    }


def idf_ni1(df, n_docs):
    """
    Normalized IDF based on the document frequency
    of the most frequent term across the corpus.

    IDF(t) = log(max_DF / DF(t))
    """
    max_df = max(df.values()) if df else 1

    return {
        term: math.log(max_df / d)
        if d > 0
        else 0.0
        for term, d in df.items()
    }


IDF_FUNCS = {
    "un": idf_un,
    "ni1": idf_ni1
}


In [8]:
# ============================================================
# Mode-aware TF-IDF vectorizer
# ============================================================

class TfidfVectorizerV2:
    def __init__(
        self,
        min_df=2,
        max_features=20000,
        tf_mode="un",
        idf_mode="un"
    ):
        if tf_mode not in TF_FUNCS:
            raise ValueError(f"Unknown tf_mode: {tf_mode}")

        if idf_mode not in IDF_FUNCS:
            raise ValueError(f"Unknown idf_mode: {idf_mode}")

        self.min_df = min_df
        self.max_features = max_features
        self.tf_mode = tf_mode
        self.idf_mode = idf_mode

        self.tf_fn = TF_FUNCS[tf_mode]
        self.idf_fn = IDF_FUNCS[idf_mode]

        self.vocab_ = {}
        self.idf_ = []

    def fit(self, corpus):
        # Document frequency over the training corpus
        df = defaultdict(int)

        for text in corpus:
            features = set(extract_features(text))

            for feature in features:
                df[feature] += 1

        # Keep features that satisfy min_df
        candidates = [
            (feature, count)
            for feature, count in df.items()
            if count >= self.min_df
        ]

        # Same vocabulary selection as Assignment 1
        candidates.sort(
            key=lambda x: x[1],
            reverse=True
        )

        if self.max_features is not None:
            candidates = candidates[:self.max_features]

        self.vocab_ = {
            feature: i
            for i, (feature, _) in enumerate(candidates)
        }

        # Calculate IDF only for selected vocabulary
        n_docs = len(corpus)

        vocab_df = {
            feature: df[feature]
            for feature in self.vocab_
        }

        idf_by_term = self.idf_fn(
            vocab_df,
            n_docs
        )

        self.idf_ = [0.0] * len(self.vocab_)

        for feature, i in self.vocab_.items():
            self.idf_[i] = idf_by_term[feature]

        return self

    def transform(self, corpus):
        vectors = []

        for text in corpus:
            counts = Counter(
                extract_features(text)
            )

            # IMPORTANT:
            # text is passed so normalized TF can use
            # the actual words in the sentence.
            tf_weights = self.tf_fn(
                counts,
                text
            )

            vec = {}

            for feature, tf_value in tf_weights.items():
                j = self.vocab_.get(feature)

                if j is None:
                    continue

                vec[j] = (
                    tf_value * self.idf_[j]
                )

            # Keep the same final L2 normalization
            # used in Assignment 1.
            norm = math.sqrt(
                sum(v * v for v in vec.values())
            ) or 1.0

            for j in vec:
                vec[j] /= norm

            vectors.append(vec)

        return vectors

    def fit_transform(self, corpus):
        self.fit(corpus)
        return self.transform(corpus)

    @property
    def n_features(self):
        return len(self.vocab_)


## Six TF-IDF combinations

The same Logistic Regression model is trained for every combination.

| # | TF | IDF |
|---|---|---|
| 1 | Unnormalized | Unnormalized |
| 2 | Total-word normalized | Unnormalized |
| 3 | Maximum-frequency normalized | Unnormalized |
| 4 | Unnormalized | Normalized |
| 5 | Total-word normalized | Normalized |
| 6 | Maximum-frequency normalized | Normalized |


In [9]:
# ============================================================
# All six combinations
# ============================================================

COMBINATIONS = [
    ("un", "un"),      # 1. Raw TF + Raw IDF
    ("nt1", "un"),     # 2. Total-word normalized TF + Raw IDF
    ("nt2", "un"),     # 3. Max-frequency normalized TF + Raw IDF
    ("un", "ni1"),     # 4. Raw TF + Normalized IDF
    ("nt1", "ni1"),    # 5. Total-word normalized TF + Normalized IDF
    ("nt2", "ni1"),    # 6. Max-frequency normalized TF + Normalized IDF
]

COMBINATION_NAMES = {
    ("un", "un"): "Unnormalized TF + Unnormalized IDF",
    ("nt1", "un"): "Total-word normalized TF + Unnormalized IDF",
    ("nt2", "un"): "Max-frequency normalized TF + Unnormalized IDF",
    ("un", "ni1"): "Unnormalized TF + Normalized IDF",
    ("nt1", "ni1"): "Total-word normalized TF + Normalized IDF",
    ("nt2", "ni1"): "Max-frequency normalized TF + Normalized IDF",
}


In [10]:
# ============================================================
# Train and evaluate all six models
# ============================================================

def run_all_combinations():
    print("Loading Lab1 dataset...")

    # Dataset is reused from Lab1.
    train_texts, train_labels = load_jsonl(TRAIN_PATH)
    val_texts, val_labels = load_jsonl(VAL_PATH)
    test_texts, test_labels = load_jsonl(TEST_PATH)

    label_set = sorted(set(train_labels))

    label2idx = {
        label: i
        for i, label in enumerate(label_set)
    }

    idx2label = {
        i: label
        for label, i in label2idx.items()
    }

    y_train = [
        label2idx[label]
        for label in train_labels
    ]

    y_val = [
        label2idx[label]
        for label in val_labels
    ]

    y_test = [
        label2idx[label]
        for label in test_labels
    ]

    results = []

    for experiment_no, (tf_mode, idf_mode) in enumerate(
        COMBINATIONS,
        start=1
    ):
        print("\n" + "=" * 70)
        print(f"EXPERIMENT {experiment_no}/6")
        print(
            COMBINATION_NAMES[
                (tf_mode, idf_mode)
            ]
        )
        print("=" * 70)

        # Build the TF-IDF vectorizer for this combination.
        vectorizer = TfidfVectorizerV2(
            min_df=2,
            max_features=20000,
            tf_mode=tf_mode,
            idf_mode=idf_mode
        )

        print("Creating TF-IDF vectors...")

        X_train = vectorizer.fit_transform(
            train_texts
        )

        X_val = vectorizer.transform(
            val_texts
        )

        X_test = vectorizer.transform(
            test_texts
        )

        print(
            "Vocabulary size:",
            vectorizer.n_features
        )

        # Same Logistic Regression settings
        # for every experiment.
        print("\nTraining Logistic Regression...")

        clf = LogisticRegression(
            n_features=vectorizer.n_features,
            n_classes=len(label_set),
            lr=0.5,
            epochs=20,
            batch_size=64,
            l2=1e-4,
            verbose=True,
            seed=0
        )

        clf.fit(
            X_train,
            y_train,
            X_val,
            y_val
        )

        # Test evaluation
        print("\nEvaluating on test set...")

        y_pred = clf.predict(X_test)

        acc = accuracy(
            y_test,
            y_pred
        )

        macro = macro_f1(
            y_test,
            y_pred,
            len(label_set)
        )

        print(
            f"Test accuracy : {acc:.4f}"
        )

        print(
            f"Test macro-F1 : {macro:.4f}"
        )

        results.append({
            "experiment": experiment_no,
            "tf_mode": tf_mode,
            "idf_mode": idf_mode,
            "name": COMBINATION_NAMES[
                (tf_mode, idf_mode)
            ],
            "test_accuracy": acc,
            "test_macro_f1": macro
        })

    return results


In [11]:
# ============================================================
# Run all 6 experiments
# ============================================================

results = run_all_combinations()


Loading Lab1 dataset...

EXPERIMENT 1/6
Unnormalized TF + Unnormalized IDF
Creating TF-IDF vectors...
Vocabulary size: 20000

Training Logistic Regression...
epoch   1/20  train_loss=2.7909  val_acc=0.8809
epoch   2/20  train_loss=2.1733  val_acc=0.9200
epoch   3/20  train_loss=1.7163  val_acc=0.9143
epoch   4/20  train_loss=1.4036  val_acc=0.9183
epoch   5/20  train_loss=1.1898  val_acc=0.9252
epoch   6/20  train_loss=1.0380  val_acc=0.9274
epoch   7/20  train_loss=0.9258  val_acc=0.9291
epoch   8/20  train_loss=0.8397  val_acc=0.9287
epoch   9/20  train_loss=0.7716  val_acc=0.9330
epoch  10/20  train_loss=0.7168  val_acc=0.9387
epoch  11/20  train_loss=0.6713  val_acc=0.9365
epoch  12/20  train_loss=0.6329  val_acc=0.9348
epoch  13/20  train_loss=0.6004  val_acc=0.9387
epoch  14/20  train_loss=0.5724  val_acc=0.9396
epoch  15/20  train_loss=0.5479  val_acc=0.9396
epoch  16/20  train_loss=0.5264  val_acc=0.9404
epoch  17/20  train_loss=0.5073  val_acc=0.9396
epoch  18/20  train_loss=0

In [12]:
# ============================================================
# Final comparison table
# ============================================================

print("\n" + "=" * 90)
print("FINAL COMPARISON — ALL 6 TF/IDF COMBINATIONS")
print("=" * 90)

print(
    f"{'#':<4}"
    f"{'TF':<10}"
    f"{'IDF':<10}"
    f"{'Accuracy':>14}"
    f"{'Macro-F1':>14}"
)

print("-" * 90)

for result in results:
    print(
        f"{result['experiment']:<4}"
        f"{result['tf_mode']:<10}"
        f"{result['idf_mode']:<10}"
        f"{result['test_accuracy']:>14.4f}"
        f"{result['test_macro_f1']:>14.4f}"
    )



FINAL COMPARISON — ALL 6 TF/IDF COMBINATIONS
#   TF        IDF             Accuracy      Macro-F1
------------------------------------------------------------------------------------------
1   un        un                0.9387        0.9384
2   nt1       un                0.9387        0.9384
3   nt2       un                0.9387        0.9384
4   un        ni1               0.9465        0.9466
5   nt1       ni1               0.9465        0.9466
6   nt2       ni1               0.9465        0.9466


In [13]:
# ============================================================
# Find the best model
# ============================================================

best_accuracy = max(
    results,
    key=lambda r: r["test_accuracy"]
)

best_macro_f1 = max(
    results,
    key=lambda r: r["test_macro_f1"]
)

print("\nBest by Test Accuracy:")
print(
    best_accuracy["name"],
    "->",
    f"{best_accuracy['test_accuracy']:.4f}"
)

print("\nBest by Test Macro-F1:")
print(
    best_macro_f1["name"],
    "->",
    f"{best_macro_f1['test_macro_f1']:.4f}"
)



Best by Test Accuracy:
Unnormalized TF + Normalized IDF -> 0.9465

Best by Test Macro-F1:
Unnormalized TF + Normalized IDF -> 0.9466


In [15]:
# ============================================================
# Reload labels and create label mappings for detailed evaluation
# ============================================================

train_texts, train_labels = load_jsonl(TRAIN_PATH)
val_texts, val_labels = load_jsonl(VAL_PATH)
test_texts, test_labels = load_jsonl(TEST_PATH)

# Get all language labels
label_set = sorted(set(train_labels))

# Language -> integer
label2idx = {
    label: i
    for i, label in enumerate(label_set)
}

# Integer -> language
idx2label = {
    i: label
    for label, i in label2idx.items()
}

# Convert labels to integers
y_train = [
    label2idx[label]
    for label in train_labels
]

y_val = [
    label2idx[label]
    for label in val_labels
]

y_test = [
    label2idx[label]
    for label in test_labels
]

print("Number of languages:", len(label_set))
print("Languages:", label_set)

Number of languages: 23
Languages: ['asm_Beng', 'ben_Beng', 'brx_Deva', 'doi_Deva', 'gom_Deva', 'guj_Gujr', 'hin_Deva', 'kan_Knda', 'kas_Arab', 'khasi', 'mai_Deva', 'mal_Mlym', 'mar_Deva', 'mni_Mtei', 'npi_Deva', 'ory_Orya', 'pan_Guru', 'san_Deva', 'santhali', 'snd_Deva', 'tam_Taml', 'tel_Telu', 'urd_Arab']


In [16]:
# ============================================================
# Detailed per-language evaluation for the best Macro-F1 model
# ============================================================

best_tf = best_macro_f1["tf_mode"]
best_idf = best_macro_f1["idf_mode"]

print("\nTraining the best Macro-F1 configuration again for")
print(best_macro_f1["name"])

best_vectorizer = TfidfVectorizerV2(
    min_df=2,
    max_features=20000,
    tf_mode=best_tf,
    idf_mode=best_idf
)

X_train_best = best_vectorizer.fit_transform(
    train_texts
)

X_val_best = best_vectorizer.transform(
    val_texts
)

X_test_best = best_vectorizer.transform(
    test_texts
)

best_clf = LogisticRegression(
    n_features=best_vectorizer.n_features,
    n_classes=len(label_set),
    lr=0.5,
    epochs=20,
    batch_size=64,
    l2=1e-4,
    verbose=True,
    seed=0
)

best_clf.fit(
    X_train_best,
    y_train,
    X_val_best,
    y_val
)

best_predictions = best_clf.predict(
    X_test_best
)

precision, recall, f1 = precision_recall_f1_per_class(
    y_test,
    best_predictions,
    len(label_set)
)

print("\n" + "=" * 55)
print("PER-LANGUAGE RESULTS FOR BEST MODEL")
print("=" * 55)

print(
    f"{'Language':<15}"
    f"{'Precision':>12}"
    f"{'Recall':>12}"
    f"{'F1':>12}"
)

for i, language in idx2label.items():
    print(
        f"{language:<15}"
        f"{precision[i]:>12.3f}"
        f"{recall[i]:>12.3f}"
        f"{f1[i]:>12.3f}"
    )



Training the best Macro-F1 configuration again for
Unnormalized TF + Normalized IDF
epoch   1/20  train_loss=2.8786  val_acc=0.8748
epoch   2/20  train_loss=2.3931  val_acc=0.9348
epoch   3/20  train_loss=1.9930  val_acc=0.9200
epoch   4/20  train_loss=1.6848  val_acc=0.9348
epoch   5/20  train_loss=1.4537  val_acc=0.9357
epoch   6/20  train_loss=1.2791  val_acc=0.9396
epoch   7/20  train_loss=1.1443  val_acc=0.9400
epoch   8/20  train_loss=1.0379  val_acc=0.9417
epoch   9/20  train_loss=0.9520  val_acc=0.9461
epoch  10/20  train_loss=0.8815  val_acc=0.9457
epoch  11/20  train_loss=0.8225  val_acc=0.9470
epoch  12/20  train_loss=0.7722  val_acc=0.9457
epoch  13/20  train_loss=0.7292  val_acc=0.9474
epoch  14/20  train_loss=0.6919  val_acc=0.9491
epoch  15/20  train_loss=0.6593  val_acc=0.9487
epoch  16/20  train_loss=0.6305  val_acc=0.9461
epoch  17/20  train_loss=0.6048  val_acc=0.9452
epoch  18/20  train_loss=0.5819  val_acc=0.9457
epoch  19/20  train_loss=0.5614  val_acc=0.9457
epo